In [8]:
import sympy as sp
from decimal import Decimal, ROUND_DOWN

In [25]:
def truncar(numero, decimales):
    """
    Función encargada de truncar un número decimal a la cantidad de decimales solicitada.

    parameters:
        numero: Número que se desea truncar.
        decimales: Cantidad de posiciones decimales que se quiere conservar.
    
    returns:
        Número truncado.
    """
    
    numero = Decimal(str(numero))
    factor = Decimal(1).scaleb(-decimales)
    return numero.quantize(factor, rounding=ROUND_DOWN)

In [10]:
x = 0.335555
x_truncado = truncar(x, 4)
print("Original:", x)
print("Truncamiento:", x_truncado)

Original: 0.335555
Truncamiento: 0.3355


In [26]:
def agregar_termino(Pn_str, coef_truncado, potencia, decimales):
    """
    Función encargada de construir y agregar un término al polinomio de Taylor.

    Recibe el polinomio construido hasta el momento, su coeficiente ya
    truncado, su potencia y la cantidad de decimales que se mostrarán.
    
    Se devuelve el polinomio ya organizado, los coeficientes en formato decimal y con el signo
    correspondiente.

    Args:
        Pn_str (str): Cadena que contiene los términos del polinomio.
        coef_truncado (Decimal): Coeficiente ya truncado.
        potencia (int): Exponente correspondiente a la variable x.
        decimales (int): Cantidad de posiciones decimales a mostrarse.

    Returns:
        str: Cadena de texto con el nuevo término ubicado en el polinomio.

    Example:
        >>> agregar_termino("", -0.906, 1, 4)
        '-0.9060*x'
    """

    numero = f"{abs(coef_truncado):.{decimales}f}"

    if potencia == 0:
        termino = numero
    elif potencia == 1:
        termino = f"{numero}*x"
    else:
        termino = f"{numero}*x**{potencia}"

    if not Pn_str:
        Pn_str = ("-" if coef_truncado < 0 else "") + termino
    else:
        signo = "-" if coef_truncado < 0 else "+"
        Pn_str += f" {signo} {termino}"

    return Pn_str

In [ ]:
def polinomio_taylor(func_str, x0, grado, decimales):
    
    """
    :func_str: string, función original
    :x0: entero, punto alrededor del cual se desarrolla el polinomio    
    :grado: entero, grado de polinomio de Taylor    
    :decimales: entero, decimales a truncar
    :return: polinomio de taylor aproximado a la función origianl, f(x) ≈ P_n(x) + R_n(x)
    """
    
    x = sp.Symbol('x')
    f = sp.sympify(func_str)

    taylor_expr = 0

    for n in range(grado + 1):
        derivada = f.diff(x, n)
        deriv_val = derivada.subs(x, x0)
        factorial_n = sp.factorial(n)
        coef = deriv_val / factorial_n
        
        #Se procedió a comentar las partes a eliminar del código

        # if coef.is_number:
        #     coef = truncar(float(coef), decimales)

        # if coef == 0:
        #     continue

        base = (x - x0)**n if x0 != 0 else x**n
        taylor_expr += coef * base

    taylor_expr = sp.expand(taylor_expr)
    
    polinomio = sp.Poly(taylor_expr, x)
    
    Pn_str = ""
    
    for (potencia,), coef in polinomio.terms():

        coef_truncado = truncar(
        Decimal(str(sp.N(coef, 30))),
        decimales
        )

        if coef_truncado == 0:
            continue

        Pn_str = agregar_termino(Pn_str, coef_truncado, potencia, decimales)
        
    
    # taylor_expr = sp.collect(taylor_expr, x)

    # Pn_str = str(taylor_expr)

    n_plus_1 = grado + 1
    factorial_np1 = sp.factorial(n_plus_1)
    #inverso_truncado = truncar(1 / float(factorial_np1), decimales)
    
    inverso_truncado = truncar(Decimal(1) / Decimal(str(factorial_np1)), decimales)

    base_error = sp.expand((x - x0)**n_plus_1)
    #Rn_str = f"( {inverso_truncado}*{func_str}^({n_plus_1})(ξ(x)) )*({base_error})"

    Rn_str = (
        f"({inverso_truncado:.{decimales}f}"
        f"*{func_str}^({n_plus_1})(ξ(x)))"
        f"*({base_error})"
    )
    
    print(f"P_n(x) ≈ {Pn_str}")
    print(f"R_n(x) ≈ {Rn_str}")

    return f"f(x) ≈ {Pn_str} + {Rn_str}"

#### Funciones válidas para función `taylor`

Funciones Trigonométricas

| Función Matemática | String válido (`func_str`) |
|--------------------|-----------------------------|
| \( sin(x) \)      | `"sin(x)"`                  |
| \( cos(x) \)      | `"cos(x)"`                  |
| \( tan(x) \)      | `"tan(x)"`                  |
| \( cot(x) \)      | `"cot(x)"`                  |
| \( sec(x) \)      | `"sec(x)"`                  |
| \( csc(x) \)      | `"csc(x)"`                  |



Funciones Exponenciales y Logarítmicas

| Función Matemática   | String válido (`func_str`) |
|----------------------|-----------------------------|
| \( e^x \)            | `"exp(x)"`                  |
| \( ln(x) \)         | `"ln(x)"`                   |
| \( log_{10}(x) \)   | `"log(x, 10)"`              |
| \( log_b(x) \)      | `"log(x, b)"` (reemplazar `b`) |



Funciones Misceláneas

| Función Matemática | String válido (`func_str`) |
|--------------------|-----------------------------|
| \( \sqrt{x} \)     | `"sqrt(x)"`                 |
| \( \|x\| \)          | `"Abs(x)"`                  |
| Polinomio típico   | `"x**2 + 3*x"`              |


In [22]:
print(polinomio_taylor("exp(x)", x0=1, grado=4, decimales=4))

P_n(x) ≈ 0.1132*x**4 + 0.000200000000000033*x**3 + 0.6793*x**2 + 0.9062*x + 1.0193
R_n(x) ≈ ( 0.0083*exp(x)^(5)(ξ(x)) )*(x**5 - 5*x**4 + 10*x**3 - 10*x**2 + 5*x - 1)
f(x) ≈ 0.1132*x**4 + 0.000200000000000033*x**3 + 0.6793*x**2 + 0.9062*x + 1.0193 + ( 0.0083*exp(x)^(5)(ξ(x)) )*(x**5 - 5*x**4 + 10*x**3 - 10*x**2 + 5*x - 1)


In [17]:
print(polinomio_taylor("exp(x)", x0=1, grado=4, decimales=4))

P_n(x) ≈ 0.1132*x**4 + 0.6795*x**2 + 0.9060*x + 1.0193
R_n(x) ≈ (0.0083*exp(x)^(5)(ξ(x)))*(x**5 - 5*x**4 + 10*x**3 - 10*x**2 + 5*x - 1)
f(x) ≈ 0.1132*x**4 + 0.6795*x**2 + 0.9060*x + 1.0193 + (0.0083*exp(x)^(5)(ξ(x)))*(x**5 - 5*x**4 + 10*x**3 - 10*x**2 + 5*x - 1)


In [18]:
print(polinomio_taylor("exp(-x)", x0=1, grado=3, decimales=4))

P_n(x) ≈ -0.0613*x**3 + 0.3678*x**2 - 0.9196*x + 0.9810
R_n(x) ≈ (0.0416*exp(-x)^(4)(ξ(x)))*(x**4 - 4*x**3 + 6*x**2 - 4*x + 1)
f(x) ≈ -0.0613*x**3 + 0.3678*x**2 - 0.9196*x + 0.9810 + (0.0416*exp(-x)^(4)(ξ(x)))*(x**4 - 4*x**3 + 6*x**2 - 4*x + 1)



### Unidad 4: Interpolación y Ajuste de Curvas

En esta unidad se estudiarán diferentes métodos numéricos
que permiten aproximar funciones mediante polinomios o
encontrar expresiones matemáticas que representen un
conjunto de datos.

Es importante distinguir entre los siguientes conceptos:

- **Interpolación:** Consiste en estimar un valor desconocido
  que se encuentra dentro de un conjunto de datos conocidos.

- **Extrapolación:** Consiste en estimar un valor desconocido
  que se encuentra fuera del intervalo de datos conocidos.

- **Ajuste de curvas:** Consiste en encontrar una función que
  represente de la mejor manera posible el comportamiento
  de un conjunto de datos.



<h4 style="text-align:center;">Polinomio de Taylor</h4>

El polinomio de Taylor permite aproximar una función mediante
un polinomio que ocupa las derivadas de dicha
función evaluadas alrededor de un punto dado, al cual lo conoceremos como $x_0$.

La función general es:

$$
f(x)=P_n(x)+R_n(x)
$$

Donde:

- $P_n(x)$: Polinomio de Taylor de orden $n$.
- $R_n(x)$: Error de truncamiento.

Su forma general es:

$$
P_n(x)=\sum_{k=0}^{n}
\frac{f^{(k)}(x_0)}{k!}(x-x_0)^k
$$

Para los cálculos a mano, resultó mejor tomar la forma ya expandida:

$$
P_n(x)=f(x_0)+f'(x_0)(x-x_0)
+\frac{f''(x_0)}{2!}(x-x_0)^2
+\cdots+\frac{f^{(n)}(x_0)}{n!}(x-x_0)^n
$$

La parte del error de truncamiento se ve así:

$$
R_n(x)=\frac{f^{(n+1)}(\xi(x))}{(n+1)!}
(x-x_0)^{n+1}
$$

Donde $\xi(x)$ corresponde a un valor que se encuentra
entre $x_0$ y $x$.
